# Download data with jstex

Selected assets or a complete product, over S3 (and HTTPS). You need `pip install "jupyterlab-jstex[s3]"` and a CDSE account (hub login, a stored session or **Sign in**).

In [ ]:
import jstex

jstex.whoami()  # 'anonymous'? Sign in in the widget below or run jstex.login()

In [ ]:
ex = jstex.Explorer()
ex

Choose a Sentinel-2 L2A item in the widget (it becomes `ex.selected_item`), or run the next cell to search from code.

In [ ]:
AOI = [10.0, 45.3, 10.4, 45.6]  # lon/lat bbox (Lake Garda, Italy)
ex.query = {
    **ex.query,
    "collections": ["sentinel-2-l2a"],
    "datetime": {"from": "2024-07-01T00:00:00Z", "to": "2024-07-31T23:59:59Z"},
    "aois": [
        {
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [AOI[0], AOI[1]],
                        [AOI[2], AOI[1]],
                        [AOI[2], AOI[3]],
                        [AOI[0], AOI[3]],
                        [AOI[0], AOI[1]],
                    ]
                ],
            },
            "selected": True,
        }
    ],
    "filters": [{"field": "eo:cloud_cover", "op": "<=", "value": 10}],
}
ex.search(wait=True)
item = ex.selected_item or next(iter(ex.results))
item.id, sorted(item.assets)

In [ ]:
def band(item, common_name):
    """The finest-resolution single-band asset of a band, by its common name (e.g. 'red', 'nir')."""
    found = []
    for key, asset in item.assets.items():
        bands = (
            asset.extra_fields.get("eo:bands") or asset.extra_fields.get("bands") or []
        )
        if any(
            common_name in (b.get("common_name"), b.get("eo:common_name"))
            for b in bands
        ):
            # finest gsd first; single-band assets before composites such as TCI
            found.append((asset.extra_fields.get("gsd") or 999, len(bands), key, asset))
    if not found:
        raise KeyError(f"no {common_name!r} band in {sorted(item.assets)}")
    _, _, key, asset = min(found)
    return key, asset

In [ ]:
from pathlib import Path

out = Path("downloads") / item.id
out.mkdir(parents=True, exist_ok=True)
for name in ("red", "nir"):
    key, asset = band(item, name)
    loc = jstex.s3.location(asset)
    target = out / loc["Key"].rsplit("/", 1)[-1]
    jstex.s3.client(asset).download_file(**loc, Filename=str(target))
    print(f"{key}: {target} ({target.stat().st_size / 1e6:.1f} MB)")

Assets with an `https://` link (their own or an `https` alternate) download over HTTPS with your access token (`jstex.access_token()`). Send the token only to hosts you trust — here, the platform's own download service.

In [ ]:
from urllib.parse import urlsplit

import requests


def https_href(asset):
    """The asset's own https:// link, or its `https` alternate."""
    if asset.href.startswith("https://"):
        return asset.href
    return ((asset.extra_fields.get("alternate") or {}).get("https") or {}).get("href")


def download_https(href, target):
    token = jstex.access_token()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    with requests.get(href, headers=headers, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(target, "wb") as fh:
            fh.writelines(resp.iter_content(1 << 20))


trusted = {"download.dataspace.copernicus.eu", "zipper.dataspace.copernicus.eu"}
for key in ("product_metadata", "SCL_60m"):  # small files; any asset key works
    asset = item.assets.get(key)
    href = asset and https_href(asset)
    if href and urlsplit(href).hostname in trusted:
        target = out / f"{key}{Path(urlsplit(asset.href).path).suffix}"
        download_https(href, target)
        print(f"{key}: {target} ({target.stat().st_size / 1e6:.1f} MB)")

The `Product` asset is the whole product as one zip (several GB): `download_https(item.assets["Product"].href, out / f"{item.id}.zip")`.

## Complete product

Every file under the product's S3 folder, keeping its layout; files already present with the same size are skipped.

In [ ]:
import os

from jstex.s3 import JstexS3Error


def s3_keys(item):
    for asset in item.assets.values():
        try:
            yield jstex.s3.location(asset)
        except JstexS3Error:
            pass  # asset without an S3 location


locs = list(s3_keys(item))
bucket = locs[0]["Bucket"]
prefix = os.path.commonprefix([loc["Key"] for loc in locs]).rsplit("/", 1)[0] + "/"
product = Path("downloads") / prefix.rstrip("/").rsplit("/", 1)[-1]
s3 = jstex.s3.client()
objects = [
    obj
    for page in s3.get_paginator("list_objects_v2").paginate(
        Bucket=bucket, Prefix=prefix
    )
    for obj in page.get("Contents", [])
]
total = sum(obj["Size"] for obj in objects)
done = 0
for obj in objects:
    target = product / obj["Key"][len(prefix) :]
    if not (target.exists() and target.stat().st_size == obj["Size"]):
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, obj["Key"], str(target))
    done += obj["Size"]
    print(f"\r{done / total:6.1%}  {target.name[:60]:<60}", end="")
print(f"\n{len(objects)} files, {total / 1e6:.0f} MB in {product}")